# Selection — Finding the k-th Element

Notes and runnable examples on **selection** (order statistics): finding the k-th smallest element — the minimum, the maximum, the median, or any rank in between — **without paying for a full sort**. The naive answer is to sort and index (`sorted(xs)[k]`), which is O(n log n). The headline result of this notebook is that you can do it in **O(n)**: a single value's rank doesn't require ordering all the others. We'll build **quickselect** (reusing **quicksort**'s partition), watch it degrade on adversarial input, fix it with **median-of-medians** for a worst-case guarantee, then survey the Python toolkit (`heapq`, `statistics`) and prove every claim against `sorted(xs)[k]`.

**Contents**

1. **The problem** — k-th smallest, the median, and the O(n) question
2. **Quickselect** — partition, but recurse into one side
3. **Worst case & the pivot** — adversarial input, then median-of-medians (BFPRT)
4. **The Python toolkit** — `heapq.nsmallest`/`nlargest`, `statistics.median`, `min`/`max`
5. **When to use what**
6. **Complexity cheat-sheet**

## 1. The problem — k-th smallest, the median, and the O(n) question

A **selection** query asks for the element of a given **rank**: the k-th smallest value (0-indexed here, so `k=0` is the minimum, `k=n-1` the maximum, `k=n//2` the median). These are the **order statistics** of the data.

The obvious solution is to **sort and index** — `sorted(xs)[k]` — which is correct but does O(n log n) work to answer a question about *one* position. (See the **sorting** notebook for why O(n log n) is the floor for comparison sorts, and the **searching** notebook for how structure buys cheap lookups.) The key insight: **finding one rank doesn't require ordering everyone else.** To know the median you only need to know which half each element falls in, not the full arrangement — and that's a job you can do in a linear number of comparisons.

So the target is **O(n)**. We'll get there with quickselect (O(n) *average*) and then median-of-medians (O(n) *guaranteed*). First, the baseline we must beat — and a reference we'll check everything against:

In [1]:
import random
random.seed(0)

def kth_by_sort(xs, k):
    """The naive O(n log n) reference: sort, then index."""
    return sorted(xs)[k]

xs = [random.randint(0, 99) for _ in range(11)]
print('data          :', xs)
print('sorted        :', sorted(xs))
print('min   (k=0)   :', kth_by_sort(xs, 0))
print('median(k=n//2):', kth_by_sort(xs, len(xs) // 2))
print('max   (k=n-1) :', kth_by_sort(xs, len(xs) - 1))

data          : [49, 97, 53, 5, 33, 65, 62, 51, 38, 61, 45]
sorted        : [5, 33, 38, 45, 49, 51, 53, 61, 62, 65, 97]
min   (k=0)   : 5
median(k=n//2): 51
max   (k=n-1) : 97


## 2. Quickselect — partition, but recurse into one side

**Quickselect** (Hoare, 1961) reuses **quicksort**'s machinery with one change that makes all the difference. Quicksort *partitions* around a pivot — everything smaller goes left, everything larger goes right — and then recurses into **both** halves. But after a partition, the pivot sits at its **final sorted position** `i`. That single fact tells you exactly where to look:

| relation | meaning | action |
|---|---|---|
| `k == i` | the pivot *is* the k-th element | return it |
| `k < i`  | the k-th element is among the smaller ones | recurse **left** only |
| `k > i`  | the k-th element is among the larger ones | recurse **right** only |

Because we discard a whole side each step instead of sorting both, the work is `n + n/2 + n/4 + … ≈ 2n` on average — **O(n)**, not O(n log n). The geometric series is the whole trick.

Below is an in-place **Lomuto partition** (one cursor `i` marks the boundary of the "seen smaller" region). We pick a **random pivot** — section 3 shows why that matters — and loop instead of recursing, so there's no stack growth:

In [2]:
import random
random.seed(0)

def quickselect(xs, k):
    """k-th smallest (0-indexed) in O(n) average. In-place Lomuto partition, random pivot."""
    a = xs[:]                       # copy so we don't clobber the caller's list
    lo, hi = 0, len(a) - 1
    while True:
        if lo == hi:                # one element left -> it's the answer
            return a[lo]
        p = random.randint(lo, hi)  # random pivot, swapped to the end
        a[p], a[hi] = a[hi], a[p]
        pivot = a[hi]
        i = lo                      # boundary: a[lo:i] are all < pivot
        for j in range(lo, hi):
            if a[j] < pivot:
                a[i], a[j] = a[j], a[i]
                i += 1
        a[i], a[hi] = a[hi], a[i]   # drop the pivot into its final slot i
        if k == i:
            return a[i]
        elif k < i:
            hi = i - 1              # answer is in the left part only
        else:
            lo = i + 1              # answer is in the right part only

demo = [7, 2, 9, 4, 1, 8, 3]
print('data  :', demo)
print('sorted:', sorted(demo))
for k in range(len(demo)):
    print(f'  k={k}: quickselect={quickselect(demo, k)}  sorted[k]={sorted(demo)[k]}')

data  : [7, 2, 9, 4, 1, 8, 3]
sorted: [1, 2, 3, 4, 7, 8, 9]
  k=0: quickselect=1  sorted[k]=1
  k=1: quickselect=2  sorted[k]=2
  k=2: quickselect=3  sorted[k]=3
  k=3: quickselect=4  sorted[k]=4
  k=4: quickselect=7  sorted[k]=7
  k=5: quickselect=8  sorted[k]=8
  k=6: quickselect=9  sorted[k]=9


**Correctness is non-negotiable**, so we hammer it against the reference over thousands of random inputs — varying sizes, duplicate-heavy ranges, and every value of `k`:

In [3]:
import random
random.seed(42)

for _ in range(5000):
    n = random.randint(1, 50)
    xs = [random.randint(-20, 20) for _ in range(n)]   # small range -> lots of ties
    k = random.randint(0, n - 1)
    assert quickselect(xs, k) == sorted(xs)[k], (xs, k)
print('quickselect matched sorted(xs)[k] on 5000 random inputs (incl. duplicates, every k) -> OK')

quickselect matched sorted(xs)[k] on 5000 random inputs (incl. duplicates, every k) -> OK


Now the payoff claim: on a large array, quickselect should beat sort-then-index. There's a catch worth naming up front — our quickselect is **pure Python** while `sorted` is **C Timsort**, so the algorithmic win (≈2n vs n log n) is heavily taxed by the constant-factor divide. We measure anyway, and phrase it as a ratio. The cleaner evidence is the **comparison count**, which is implementation-independent:

In [4]:
import random, timeit
random.seed(1)

big = [random.random() for _ in range(200_000)]
kmed = len(big) // 2

t_qs   = timeit.timeit(lambda: quickselect(big, kmed), number=5) / 5
t_sort = timeit.timeit(lambda: sorted(big)[kmed],       number=5) / 5
print(f'quickselect (pure Python): {t_qs*1000:6.1f} ms')
print(f'sorted()[k] (C Timsort)  : {t_sort*1000:6.1f} ms')
print(f'-> sort/quickselect ratio: {t_sort/t_qs:.2f}x (quickselect wins even across the Python/C gap)')

# Comparison count: the implementation-independent proof that quickselect is linear.
def quickselect_comps(xs, k):
    a = xs[:]; lo, hi = 0, len(a) - 1; comps = 0
    while True:
        if lo == hi: return comps
        p = random.randint(lo, hi); a[p], a[hi] = a[hi], a[p]
        pivot = a[hi]; i = lo
        for j in range(lo, hi):
            comps += 1
            if a[j] < pivot: a[i], a[j] = a[j], a[i]; i += 1
        a[i], a[hi] = a[hi], a[i]
        if k == i: return comps
        elif k < i: hi = i - 1
        else: lo = i + 1

n = len(big)
c = quickselect_comps(big, kmed)
print(f'\ncomparisons: quickselect ~{c/n:.1f}n  vs  a full sort ~n*log2(n) = {n.bit_length():.0f}n')

quickselect (pure Python):   18.0 ms
sorted()[k] (C Timsort)  :   22.3 ms
-> sort/quickselect ratio: 1.24x (quickselect wins even across the Python/C gap)

comparisons: quickselect ~4.8n  vs  a full sort ~n*log2(n) = 18n


## 3. Worst case & the pivot — adversarial input, then median-of-medians

Quickselect is O(n) *on average*, but a **bad pivot** wrecks it. If every partition peels off just one element instead of roughly halving, the work becomes `n + (n-1) + (n-2) + … = O(n²)` — exactly the same cliff as **quicksort**'s worst case. The textbook trap: a pivot that's always the **smallest (or largest) remaining** element.

It's easy to trigger with a **fixed** pivot (e.g. always the last element) fed **already-sorted** input. Let's count comparisons to see the quadratic blow-up directly:

In [5]:
def quickselect_fixed_comps(xs, k):
    """Quickselect with a FIXED last-element pivot -> O(n^2) on sorted input."""
    a = xs[:]; lo, hi = 0, len(a) - 1; comps = 0
    while True:
        if lo == hi: return comps
        pivot = a[hi]               # FIXED pivot, no randomness
        i = lo
        for j in range(lo, hi):
            comps += 1
            if a[j] < pivot: a[i], a[j] = a[j], a[i]; i += 1
        a[i], a[hi] = a[hi], a[i]
        if k == i: return comps
        elif k < i: hi = i - 1
        else: lo = i + 1

n = 2000
adversarial = list(range(n))                       # already sorted -> worst case for last-pivot
bad  = quickselect_fixed_comps(adversarial, 0)     # find the minimum
good = quickselect_comps(adversarial, 0)           # random pivot from section 2
print(f'fixed pivot,  sorted input: {bad:>8} comparisons   (~n^2/2 = {n*n//2})')
print(f'random pivot, sorted input: {good:>8} comparisons   (~a few n = {n})')
print(f'-> the random pivot is {bad/good:.0f}x cheaper here; that is the whole point of randomizing')

fixed pivot,  sorted input:  1999000 comparisons   (~n^2/2 = 2000000)
random pivot, sorted input:     2301 comparisons   (~a few n = 2000)
-> the random pivot is 869x cheaper here; that is the whole point of randomizing


A **random pivot** makes the O(n²) case astronomically unlikely (an adversary would have to guess the RNG), and it's what production code uses. But "unlikely" isn't "never" — for a **hard guarantee** you need a pivot that's provably good.

**Median-of-medians (BFPRT, 1973)** delivers exactly that. The idea is gorgeously self-referential: to pick a good pivot, *recursively select* one.

1. Split the array into groups of **5**.
2. Find each group's median (trivial — 5 elements).
3. **Recursively** find the median *of those medians* — that's the pivot.

That pivot is guaranteed to be greater than at least ~3/10 of the elements and less than at least ~3/10, so each partition discards a constant fraction. The recurrence `T(n) = T(n/5) + T(7n/10) + O(n)` solves to **O(n) worst case** — linear *with a guarantee*. The famous catch: the constant factor is large, so in practice it's usually **slower** than plain randomized quickselect. We'll measure that.

In [6]:
def mom_select(xs, k):
    """Median-of-medians selection (BFPRT): O(n) GUARANTEED worst case."""
    def select(a, k):
        if len(a) <= 5:
            return sorted(a)[k]
        # median of each group of 5
        medians = [sorted(a[i:i + 5])[len(a[i:i + 5]) // 2]
                   for i in range(0, len(a), 5)]
        pivot = select(medians, len(medians) // 2)     # the recursive pivot choice
        lows  = [x for x in a if x < pivot]
        highs = [x for x in a if x > pivot]
        eqs   = len(a) - len(lows) - len(highs)        # count of values == pivot
        if k < len(lows):
            return select(lows, k)
        elif k < len(lows) + eqs:
            return pivot
        else:
            return select(highs, k - len(lows) - eqs)
    return select(list(xs), k)

# correctness over many random inputs, incl. duplicate-heavy and every k
import random
random.seed(7)
for _ in range(3000):
    n = random.randint(1, 60)
    xs = [random.randint(-30, 30) for _ in range(n)]
    k = random.randint(0, n - 1)
    assert mom_select(xs, k) == sorted(xs)[k], (xs, k)
print('median-of-medians matched sorted(xs)[k] on 3000 random inputs -> OK')

median-of-medians matched sorted(xs)[k] on 3000 random inputs -> OK


Now the counter-intuitive timing: the algorithm with the *better worst-case guarantee* is the *slower* one in the average case. All that median-of-5 bookkeeping is pure overhead when a random pivot would have been fine:

In [7]:
import random, timeit
random.seed(2)

data = [random.random() for _ in range(50_000)]
kmed = len(data) // 2

t_mom = timeit.timeit(lambda: mom_select(data, kmed),   number=3) / 3
t_qs  = timeit.timeit(lambda: quickselect(data, kmed),  number=3) / 3
print(f'median-of-medians (guaranteed O(n)): {t_mom*1000:6.1f} ms')
print(f'randomized quickselect (avg O(n))  : {t_qs*1000:6.1f} ms')
print(f'-> median-of-medians is ~{t_mom/t_qs:.1f}x SLOWER despite the better guarantee (big constant)')

median-of-medians (guaranteed O(n)):    9.4 ms
randomized quickselect (avg O(n))  :    3.3 ms
-> median-of-medians is ~2.9x SLOWER despite the better guarantee (big constant)


## 4. The Python toolkit — `heapq`, `statistics`, and friends

You rarely hand-write selection in Python. The stdlib covers the common cases, and each has a sweet spot:

| Tool | Returns | Cost | Best when |
|---|---|---|---|
| `min(xs)` / `max(xs)` | one extreme | O(n) | you want just the min or max |
| `heapq.nsmallest(k, xs)` / `nlargest(k, xs)` | the k extremes, **sorted** | O(n log k) | k ≪ n and you want the k items |
| `statistics.median(xs)` / `median_low` / `median_high` | the median | O(n log n)¹ | you want the median, readable |
| `sorted(xs)[k]` | the k-th | O(n log n) | one-off, small, or you need several ranks |

<sub>¹ `statistics.median` sorts internally.</sub>

The star is **`heapq.nsmallest`/`nlargest`**: they keep a **size-k heap**, so the work is **O(n log k)** — when `k` is tiny this is nearly linear and crushes a full sort. They also return the results **already sorted** (descending for `nlargest`, ascending for `nsmallest`), which is exactly the *top-k* problem from the **heaps** notebook and the **top-k & k-way-merge** pattern:

In [8]:
import heapq, statistics

data = [9, 1, 8, 2, 7, 3, 6, 4, 5]
print('nsmallest(3):', heapq.nsmallest(3, data), '  (ascending, ready to use)')
print('nlargest(3) :', heapq.nlargest(3, data),  '  (descending)')

# both take key= and work on any iterable, just like sorted()
words = ['fig', 'apple', 'kiwi', 'banana', 'pear']
print('nlargest(2, key=len):', heapq.nlargest(2, words, key=len))

# the median family
odd, even = [3, 1, 4, 1, 5], [3, 1, 4, 1, 5, 9]
print('\nmedian(odd) :', statistics.median(odd), '   (the middle value)')
print('median(even):', statistics.median(even), '  (averages the two middles -> can be non-integer)')
print('median_low  :', statistics.median_low(even), '   median_high:', statistics.median_high(even),
      '  (pick an ACTUAL element, no averaging)')

nsmallest(3): [1, 2, 3]   (ascending, ready to use)
nlargest(3) : [9, 8, 7]   (descending)
nlargest(2, key=len): ['banana', 'apple']

median(odd) : 3    (the middle value)
median(even): 3.5   (averages the two middles -> can be non-integer)
median_low  : 3    median_high: 4   (pick an ACTUAL element, no averaging)


Two gotchas worth internalizing:

- **`nsmallest(1)` is not the way to get the minimum** — `min()` is simpler and a touch faster (no heap setup). `heapq` even special-cases `k=1` to fall back to `min`/`max`, so you gain nothing by reaching for it.
- **`statistics.median` of an even-length list averages the two middle values**, which can produce a non-integer (or a `float` from `int` inputs). When you need a value that actually *exists* in the data, use `median_low` / `median_high`.

Now the central trade-off: **`nsmallest(k)` vs `sorted(xs)[:k]` as k grows.** The heap's O(n log k) dominates for small k but loses its edge as k approaches n — at which point you're effectively sorting everything anyway, and C Timsort wins:

In [9]:
import heapq, timeit, random
random.seed(0)

N = 200_000
data = [random.random() for _ in range(N)]
print(f'N = {N:,}   nsmallest(k)  vs  sorted(data)[:k]')
print(f"  {'k':>8} {'nsmallest':>12} {'sorted[:k]':>12}   winner")
for k in (5, 100, 5_000, N // 2):
    t_heap = timeit.timeit(lambda: heapq.nsmallest(k, data), number=3) / 3
    t_sort = timeit.timeit(lambda: sorted(data)[:k],         number=3) / 3
    winner = 'heapq' if t_heap < t_sort else 'sorted'
    print(f'  {k:>8} {t_heap*1000:>10.1f}ms {t_sort*1000:>10.1f}ms   {winner}')

# nsmallest(1) vs min: min wins
t1 = timeit.timeit(lambda: heapq.nsmallest(1, data), number=10) / 10
t2 = timeit.timeit(lambda: min(data),                number=10) / 10
print(f'\nnsmallest(1): {t1*1000:.2f} ms   min(): {t2*1000:.2f} ms   -> use min() for a single extreme')

N = 200,000   nsmallest(k)  vs  sorted(data)[:k]
         k    nsmallest   sorted[:k]   winner
         5        1.1ms       20.4ms   heapq


       100        1.4ms       22.2ms   heapq
      5000        7.5ms       22.6ms   heapq


    100000       79.0ms       22.2ms   sorted

nsmallest(1): 0.87 ms   min(): 0.87 ms   -> use min() for a single extreme


The crossover is the practical takeaway: **`nsmallest`/`nlargest` win big when `k` is small relative to `n`** (the size-k heap barely grows), but once `k` is a large fraction of `n` you should just `sorted()[:k]` — Timsort's C speed beats a heap that now holds half the array. And for a *single* rank that isn't an extreme (the true median, say), neither of these is optimal — that's quickselect's territory.

## 5. When to use what

| You need... | Reach for | Why |
|---|---|---|
| Just the min or max | `min(xs)` / `max(xs)` | O(n), simplest, fastest |
| A *single* k-th element (e.g. the median) | **quickselect** | O(n) average, beats sort's O(n log n) |
| The k smallest / largest **items**, k ≪ n | `heapq.nsmallest` / `nlargest` | O(n log k), returns them sorted |
| A worst-case **guarantee** of O(n) | **median-of-medians** | provably linear (but big constant) |
| The median, readably | `statistics.median` / `median_low` | clear intent; sorts internally |
| Several different ranks, or small/one-off n | `sorted(xs)[k]` | sort once, index many times |
| k near n/2 (a large fraction) | `sorted(xs)[:k]` | you're sorting anyway; C Timsort wins |

**Rules of thumb.** For a single rank in production Python, **quickselect** is the right algorithm but you'll rarely beat C `sorted` for small inputs — reach for it only on genuinely large arrays. **`heapq.nsmallest`/`nlargest`** is the everyday workhorse for top-k. **Median-of-medians** is for when an adversary controls the input and you need the guarantee; otherwise its constant factor makes it the slowest option. And if you need *more than one* rank, sort once and index — the O(n log n) amortizes away.

## 6. Complexity cheat-sheet

| Method | Best | Average | Worst | Space | Notes |
|---|:---:|:---:|:---:|:---:|---|
| `min` / `max` | O(n) | O(n) | O(n) | O(1) | single extreme only |
| **Quickselect** (random pivot) | O(n) | O(n) | O(n²) | O(1)¹ | bad pivot ⇒ quadratic; reuses **quicksort**'s partition |
| **Median-of-medians** (BFPRT) | O(n) | O(n) | O(n) | O(n) | guaranteed linear, large constant ⇒ slow in practice |
| `heapq.nsmallest`/`nlargest` (k items) | O(n) | O(n log k) | O(n log k) | O(k) | size-k heap; returns sorted; great when k ≪ n |
| `statistics.median` | O(n log n) | O(n log n) | O(n log n) | O(n) | sorts internally |
| `sorted(xs)[k]` (sort-then-index) | O(n) | O(n log n) | O(n log n) | O(n) | Timsort; reuse for many ranks |

<sub>¹ The iterative quickselect above is O(1) extra space (it loops, not recurses) beyond the O(n) input copy. n = number of elements; k = the rank or the number of items requested. "Quadratic worst case" for quickselect is what median-of-medians exists to eliminate; for n items in a bounded integer range, a **counting**-style pass (see the **sorting** notebook) can find any rank in O(n) without comparisons at all.</sub>